# 2013 Round 2 Theory And Practice - Data Analysis

This notebook analyzes and computes answers to 15 multiple-choice questions about financial modeling, Excel, and VBA.

In [ ]:
import math
from datetime import date

# Read all question files
questions = {}
for i in range(1, 16):
    with open(f'/workspace/data/question{i}.txt', 'r') as f:
        questions[f'question{i}'] = f.read().strip()
    print(f'Question {i}: {questions[f"question{i}"][:80]}...')

In [ ]:
# ============================================================
# QUESTION 1: Diminishing balance (double declining) depreciation
# $18,000 asset, 5-year effective life, book value after 3 years
# ============================================================

cost = 18000
life = 5
rate = 2 / life  # Double declining rate = 40%

book_value = cost
for year in range(3):
    depreciation = book_value * rate
    book_value -= depreciation
    print(f'Year {year+1}: Depreciation = ${depreciation:.2f}, Book Value = ${book_value:.2f}')

# book_value = 18000 * (1-0.4)^3 = 18000 * 0.216 = 3888
# Closest to $4,000 => option C
options_q1 = {'A': 12000, 'B': 8500, 'C': 4000, 'D': 2000}
q1_answer = min(options_q1, key=lambda k: abs(book_value - options_q1[k]))
print(f'\nComputed book value: ${book_value:.2f}')
print(f'Q1 Answer: {q1_answer}')

In [ ]:
# ============================================================
# QUESTION 2: Free Cash Flow for the Firm (FCFF)
# Which does NOT impact FCFF?
# ============================================================
# FCFF = EBIT(1-t) + D&A - CapEx - Change in NWC
# a. Changes in NWC -> impacts FCFF
# b. Loan amortization payment -> this is a financing activity, NOT affecting FCFF
# c. Interest rate increases -> affects interest expense but FCFF is before debt service;
#    however interest affects tax shield. In standard FCFF, interest is added back.
#    But FCFF is calculated before interest, so interest rate changes on debt
#    don't directly impact FCFF (they affect FCFE).
# d. Reduction in tax rate -> affects FCFF through EBIT(1-t)
#
# Loan amortization (principal repayment) is a financing cash flow, not operating.
# It does NOT impact FCFF.

q2_answer = 'B'
print(f'Q2 Answer: {q2_answer}')
print('Loan amortization payment is a financing activity, not affecting FCFF.')

In [ ]:
# ============================================================
# QUESTION 3: Compound interest monthly vs quarterly
# $10,000 at 7.5% per annum, 30/360 basis
# ============================================================

principal = 10000
annual_rate = 0.075

# Monthly compounding: 12 periods
monthly_rate = annual_rate / 12
monthly_balance = principal * (1 + monthly_rate) ** 12

# Quarterly compounding: 4 periods
quarterly_rate = annual_rate / 4
quarterly_balance = principal * (1 + quarterly_rate) ** 4

difference = monthly_balance - quarterly_balance
print(f'Monthly balance: ${monthly_balance:.6f}')
print(f'Quarterly balance: ${quarterly_balance:.6f}')
print(f'Difference: ${difference:.6f}')

# Difference ~ $5.35, closest to $5.00 => option B
options_q3 = {'A': 2.50, 'B': 5.00, 'C': 15.00, 'D': 25.00}
q3_answer = min(options_q3, key=lambda k: abs(difference - options_q3[k]))
print(f'Q3 Answer: {q3_answer}')

In [ ]:
# ============================================================
# QUESTION 4: Fair Value vs Market Value
# ============================================================
# Fair Value and Market Value are NOT the same.
# Fair Value may include synergistic or strategic value to a specific buyer
# that the open market doesn't recognize, making Fair Value potentially
# greater than Market Value.
# Answer: A

q4_answer = 'A'
print(f'Q4 Answer: {q4_answer}')
print('Fair Value may include additional value a specific buyer sees, making it >= Market Value.')

In [ ]:
# ============================================================
# QUESTION 5: Excel number format
# Display: 0 as "-", 10.22 as "10.22x", -1.3 as "(1.3x)", text as "Error"
# ============================================================
# Excel custom number format has 4 sections: positive;negative;zero;text
#
# Option d: #,###.##x;(#,###.##x);"-";"Error"
# - Positive: #,###.##x -> 10.22x (## shows up to 2 decimal places)
# - Negative: (#,###.##x) -> (1.3x)
# - Zero: "-" -> displays dash
# - Text: "Error" -> displays Error
#
# Wait - let me analyze more carefully:
# Option a: #,##0.0x -> always shows at least 1 decimal. 10.22 would show as 10.2x (only 1 forced decimal with 0)
#   Actually, the format #,##0.0 forces 1 decimal place minimum. For 10.22, it might show 10.2x (rounding).
#   But the question says it should show 10.22x. So we need ##.
# Option d: #,###.##x -> ## means optional digits. For 10.22, shows 10.22x. For -1.3, shows (1.3x).
#   Zero section: "-", Text section: "Error" - all match!

q5_answer = 'D'
print(f'Q5 Answer: {q5_answer}')
print('Format #,###.##x;(#,###.##x);"-";"Error" correctly handles all cases.')

In [ ]:
# ============================================================
# QUESTION 6: OFFSET(A1,4,3,2,3)
# ============================================================
# OFFSET(reference, rows, cols, height, width)
# Starting from A1, move 4 rows down and 3 cols right -> D5
# Then return a range of height=2, width=3
# So: D5:F6

# A1 + 4 rows = row 5, A1 + 3 cols = col D
# Height 2, Width 3 -> D5:F6
start_row = 1 + 4  # = 5
start_col = 1 + 3  # = 4 (D)
end_row = start_row + 2 - 1  # = 6
end_col = start_col + 3 - 1  # = 6 (F)

col_letters = {1:'A', 2:'B', 3:'C', 4:'D', 5:'E', 6:'F', 7:'G'}
result_range = f"{col_letters[start_col]}{start_row}:{col_letters[end_col]}{end_row}"
print(f'OFFSET(A1,4,3,2,3) returns range: {result_range}')

q6_answer = 'A'
print(f'Q6 Answer: {q6_answer}')

In [ ]:
# ============================================================
# QUESTION 7: Commitment fee calculation
# $30m facility, 1% p.a. on daily undrawn balance
# Drawdowns: $10m on 1 Feb, 1 Mar, 1 Apr 2013
# ============================================================

facility = 30_000_000

# Period 1: 1 Jan to 31 Jan (31 days), undrawn = $30m
d1 = (date(2013, 2, 1) - date(2013, 1, 1)).days  # 31
# Period 2: 1 Feb to 28 Feb (28 days), undrawn = $20m
d2 = (date(2013, 3, 1) - date(2013, 2, 1)).days  # 28
# Period 3: 1 Mar to 31 Mar (31 days), undrawn = $10m
d3 = (date(2013, 4, 1) - date(2013, 3, 1)).days  # 31

fee_rate = 0.01

fee1 = 30_000_000 * fee_rate * d1 / 365
fee2 = 20_000_000 * fee_rate * d2 / 365
fee3 = 10_000_000 * fee_rate * d3 / 365

total_fee = fee1 + fee2 + fee3
print(f'Period 1: {d1} days, undrawn=$30m, fee=${fee1:,.2f}')
print(f'Period 2: {d2} days, undrawn=$20m, fee=${fee2:,.2f}')
print(f'Period 3: {d3} days, undrawn=$10m, fee=${fee3:,.2f}')
print(f'Total commitment fee: ${total_fee:,.2f}')

# ~$42,082 - closest to $50,000 => option B
options_q7 = {'A': 5000, 'B': 50000, 'C': 150000, 'D': 500000}
q7_answer = min(options_q7, key=lambda k: abs(total_fee - options_q7[k]))
print(f'Q7 Answer: {q7_answer}')

In [ ]:
# ============================================================
# QUESTION 8: RANDBETWEEN(1,10) equivalent
# ============================================================
# RAND() returns uniform [0, 1)
# a. =INT(10*RAND())+1
#    10*RAND() in [0, 10), INT gives 0..9, +1 gives 1..10
#    Each integer has probability 1/10. CORRECT!
# b. =ROUND(9*RAND(),0)+1
#    9*RAND() in [0,9), ROUND: 0 for [0,0.5), 1 for [0.5,1.5),...
#    0 has half the probability -> uneven
# c. =ROUND(10*RAND(),0)
#    Gives 0..10, 11 values, not 1..10
# d. =ROUND(10*RAND(),0)+1
#    Gives 1..11, not 1..10

q8_answer = 'A'
print(f'Q8 Answer: {q8_answer}')
print('INT(10*RAND())+1 produces integers 1-10 with equal probability.')

In [ ]:
# ============================================================
# QUESTION 9: Invalid Excel Defined Name
# ============================================================
# Excel Defined Names rules:
# - Cannot be a single letter that is also a cell reference
# - R and C are special in R1C1 reference style
# - Single letters like R and C are reserved
# - However, in this context, "R" and "C" are problematic
# - SUM can be used as a defined name (though it shadows the function)
# - ME is valid
# - N is a valid function name, but can be used as defined name
# - R conflicts with R1C1 notation
# Actually: In Excel, you CAN use SUM, ME, N as defined names.
# But 'R' is not valid because it's used in R1C1 reference style.
# Wait - actually 'C' single letter is also problematic.
# The answer is C (meaning option c, which is "R")

q9_answer = 'C'
print(f'Q9 Answer: {q9_answer}')
print('R is not valid as a Defined Name due to R1C1 reference style conflict.')

In [ ]:
# ============================================================
# QUESTION 10: Invalid VBA range reference
# ============================================================
# a. [C5:E5] -> Valid shorthand for Range("C5:E5")
# b. Range("C5:E5") -> Valid standard syntax
# c. ["C5:E5"] -> NOT valid. Square bracket notation doesn't use quotes inside.
#    The square bracket is already the shorthand; adding quotes makes it invalid.
# d. Range(Cells(5,3),Cells(5,5)) -> Valid

q10_answer = 'C'
print(f'Q10 Answer: {q10_answer}')
print('["C5:E5"] is not valid - square bracket shorthand does not use quotes inside.')

In [ ]:
# ============================================================
# QUESTION 11: XOR logic with AND/NOT
# Condition (i): A1=1 AND A2=1
# Condition (ii): A3=0 AND A4=0
# Return TRUE only when EXACTLY ONE condition is satisfied (XOR)
# ============================================================
# d. =XOR(AND(A1,A2),AND(NOT(A3),NOT(A4)))
#    AND(A1,A2) = TRUE when both A1=1 and A2=1 (condition i)
#    AND(NOT(A3),NOT(A4)) = TRUE when A3=0 and A4=0 (condition ii)
#    XOR returns TRUE when exactly one of these is TRUE

# Let's verify with truth table for edge cases
def check_formula_d(a1, a2, a3, a4):
    cond_i = (a1 == 1) and (a2 == 1)
    cond_ii = (a3 == 0) and (a4 == 0)
    return cond_i != cond_ii  # XOR

# Test cases:
test_cases = [
    (1, 1, 0, 0, False),  # Both conditions -> FALSE
    (1, 1, 1, 0, True),   # Only condition i -> TRUE
    (0, 1, 0, 0, True),   # Only condition ii -> TRUE
    (0, 0, 1, 1, False),  # Neither condition -> FALSE
]
for a1, a2, a3, a4, expected in test_cases:
    result = check_formula_d(a1, a2, a3, a4)
    status = 'PASS' if result == expected else 'FAIL'
    print(f'A1={a1},A2={a2},A3={a3},A4={a4}: {result} (expected {expected}) [{status}]')

q11_answer = 'D'
print(f'\nQ11 Answer: {q11_answer}')

In [ ]:
# ============================================================
# QUESTION 12: Excel function for 1D sequential array without other functions
# ============================================================
# a. OFFSET - needs a reference, returns a range but not inherently sequential
# b. ROW - ROW(1:10) returns {1;2;3;4;5;6;7;8;9;10} - sequential array!
#    ROW can generate a sequential array by itself
# c. INDEX - retrieves values from an array, doesn't generate sequences
# d. COLUMNS - returns number of columns, not a sequence

q12_answer = 'B'
print(f'Q12 Answer: {q12_answer}')
print('ROW() can generate a one-dimensional sequential array without other functions.')

In [ ]:
# ============================================================
# QUESTION 13: VBA poor programming practice
# ============================================================
# a. If Then Goto - considered poor practice (spaghetti code, hard to maintain)
# b. If Then Else - standard control flow, good practice
# c. Do Loop Until - standard loop, good practice
# d. For Next - standard loop, good practice

q13_answer = 'A'
print(f'Q13 Answer: {q13_answer}')
print('If Then Goto leads to spaghetti code and is considered poor programming practice.')

In [ ]:
# ============================================================
# QUESTION 14: Volatile function in Excel
# ============================================================
# Volatile functions recalculate every time any cell changes:
# a. MATCH - NOT volatile
# b. ROWS - NOT volatile
# c. AREAS - NOT volatile
# d. INDIRECT - VOLATILE! It recalculates on every change.

q14_answer = 'D'
print(f'Q14 Answer: {q14_answer}')
print('INDIRECT is a volatile function that recalculates on every worksheet change.')

In [ ]:
# ============================================================
# QUESTION 15: Recalculate ONLY active worksheet
# ============================================================
# Keyboard shortcut: Shift+F9 recalculates only the active sheet
# VBA: ActiveSheet.Calculate also recalculates only the active sheet
# Both methods are available.

q15_answer = 'C'
print(f'Q15 Answer: {q15_answer}')
print('Both Shift+F9 and ActiveSheet.Calculate can recalculate only the active worksheet.')

In [ ]:
# ============================================================
# FINAL ANSWERS
# ============================================================

answers = {
    'question1': q1_answer,
    'question2': q2_answer,
    'question3': q3_answer,
    'question4': q4_answer,
    'question5': q5_answer,
    'question6': q6_answer,
    'question7': q7_answer,
    'question8': q8_answer,
    'question9': q9_answer,
    'question10': q10_answer,
    'question11': q11_answer,
    'question12': q12_answer,
    'question13': q13_answer,
    'question14': q14_answer,
    'question15': q15_answer,
}

print('\n=== Final Answers ===')
for k, v in sorted(answers.items(), key=lambda x: int(x[0].replace('question', ''))):
    print(f'{k}: {v}')